# 01 — Khai phá dữ liệu (EDA)

Mục tiêu: hiểu **quy mô, schema, chất lượng và độ dài** trước khi chuẩn bị dữ liệu.
FineWeb-Edu và OpenWebMath dùng cho pretrain; **chỉ GSM8K train dùng cho SFT**.
Không khảo sát nội dung GSM8K test để chọn bộ lọc hoặc cấu hình.

Notebook chạy trên CPU. Quy mô nguồn lấy từ metadata tại revision cố định; nếu metadata thiếu thì ghi `None`, không suy ra tổng quy mô từ mẫu.
Thống kê chất lượng pretrain chỉ áp dụng cho mẫu có shuffle giới hạn bộ đệm, chưa phải mẫu ngẫu nhiên đều của toàn corpus.


In [1]:
from pathlib import Path
import importlib.util, json, subprocess, sys
REPO_URL = 'https://github.com/ngochhuy/llm-reasoning.git'
REPO_REF = 'main'  # Sau khi chốt mã, thay bằng commit SHA.
ON_KAGGLE = Path('/kaggle/working').exists()
REPO_DIR = Path('/kaggle/working/llm-reasoning-repo') if ON_KAGGLE else next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'src' / 'llm_reasoning').exists()), Path.cwd())
if ON_KAGGLE and not (REPO_DIR / 'src' / 'llm_reasoning').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
if ON_KAGGLE:
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)
if (REPO_DIR / '.git').exists():
    print('Code commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())
sys.path.insert(0, str(REPO_DIR / 'src'))
if importlib.util.find_spec('datasets') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets>=3,<5'], check=True)
from datasets import load_dataset, load_dataset_builder
from huggingface_hub import HfApi
from llm_reasoning.config import make_config
config = make_config('target', runtime='kaggle' if ON_KAGGLE else 'local')
print('Python:', sys.version.split()[0], 'artifact root:', config.paths.root)

Code commit: bbdcf4938f4db3d0edcd3990eb6989ef063afe89


d:\WORKSPACE\Project\llm-reasoning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.15 artifact root: D:\WORKSPACE\Project\llm-reasoning\artifacts


## 1. Quy mô nguồn và phạm vi khảo sát

Phân biệt ba đại lượng: **quy mô nguồn**, **ngân sách dự án** (80M/20M token), và **số mẫu thực sự khảo sát**.
FineWeb-Edu dùng subset `sample-10BT`, không phải toàn bộ FineWeb-Edu.
Số token nguồn dùng tokenizer khác nên không đồng nhất với số token BPE của dự án.


In [2]:
import html, math, statistics
from collections import Counter
from IPython.display import display, HTML
from llm_reasoning.data_prep import content_key
from llm_reasoning.sft_data import extract_gsm_answer

SAMPLE_SIZE = 1000
SHUFFLE_BUFFER = 5000
SEED = 42
assert SAMPLE_SIZE > 0 and SHUFFLE_BUFFER > 0
ROOT = Path('/kaggle/working/llm-reasoning') if ON_KAGGLE else REPO_DIR / 'artifacts'
EDA_DIR = ROOT / 'eda'
EDA_DIR.mkdir(parents=True, exist_ok=True)

def table(rows):
    if not rows:
        return
    keys = list(rows[0])
    heading = ''.join('<th style="padding:6px">' + html.escape(str(k)) + '</th>' for k in keys)
    body = ''.join('<tr>' + ''.join('<td style="padding:6px">' + html.escape(str(row.get(k))) + '</td>' for k in keys) + '</tr>' for row in rows)
    display(HTML('<table border="1"><thead><tr>' + heading + '</tr></thead><tbody>' + body + '</tbody></table>'))

sources = {
    'fineweb': (config.data.fineweb_repo, 'sample-10BT', 'text'),
    'math': (config.data.math_repo, None, 'text'),
    'gsm8k': (config.data.gsm8k_repo, 'main', 'question'),
}
api = HfApi()
revisions, inventory, samples = {}, [], {}
for label, (repo, subset, field) in sources.items():
    revision = api.repo_info(repo_id=repo, repo_type='dataset').sha
    revisions[label] = revision
    # Builder reads metadata; it does not download/prepare the full corpus.
    builder = load_dataset_builder(repo, name=subset, revision=revision)
    info = builder.info
    inventory.append({
        'source': label, 'repository': repo, 'subset': builder.config.name,
        'revision': revision,
        'split_rows_metadata': {k: v.num_examples for k, v in (info.splits or {}).items()},
        'download_GiB_metadata': round(info.download_size / 2**30, 3) if info.download_size is not None else None,
        'schema': str(info.features),
        'project_budget': '80M BPE tokens' if label == 'fineweb' else '20M BPE tokens' if label == 'math' else 'train minus 500 validation questions',
    })
    if label == 'gsm8k':
        samples[label] = list(load_dataset(repo, name=subset, split='train', revision=revision))
    else:
        stream = load_dataset(repo, name=subset, split='train', revision=revision, streaming=True)
        samples[label] = list(stream.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER).take(SAMPLE_SIZE))
table(inventory)
table([{'source': k, 'examined_rows': len(v), 'scope': 'all train' if k == 'gsm8k' else 'buffer-shuffled sample'} for k, v in samples.items()])


source,repository,subset,revision,split_rows_metadata,download_GiB_metadata,schema,project_budget
fineweb,HuggingFaceFW/fineweb-edu,sample-10BT,87f09149ef4734204d70ed1d046ddc9ca3f2b8f9,{},None,None,80M BPE tokens
math,open-web-math/open-web-math,default,fde8ef8de2300f5e778f56261843dab89f230815,{'train': 6315233},15.246,"{'url': Value('string'), 'text': Value('string'), 'date': Value('string'), 'metadata': Value('string')}",20M BPE tokens
gsm8k,openai/gsm8k,main,740312add88f781978c0658806c59bc2815b9866,"{'train': 7473, 'test': 1319}",0.003,"{'question': Value('string'), 'answer': Value('string')}",train minus 500 validation questions


source,examined_rows,scope
fineweb,1000,buffer-shuffled sample
math,1000,buffer-shuffled sample
gsm8k,7473,all train


## 2. Dữ liệu thiếu, trùng và độ dài văn bản

Đếm ô thiếu/rỗng, trùng theo hash văn bản chuẩn hóa, ký tự thay thế Unicode và marker HTML còn sót.
HTML marker chỉ là tín hiệu cần xem mẫu, không phải kết luận một tài liệu lỗi.
Với GSM8K, kiểm tra cả câu hỏi và lời giải, tỷ lệ đọc được đáp số `####`.


In [3]:
import re

def length_stats(values):
    values = sorted(values)
    if not values:
        return {'count': 0}
    return {'count': len(values), 'min': values[0], 'mean': round(statistics.mean(values), 2),
            'p50': values[(len(values)-1)//2], 'p95': values[int((len(values)-1)*.95)],
            'p99': values[int((len(values)-1)*.99)], 'max': values[-1]}

quality, lengths, text_lengths = [], [], {}
for label, (_, _, field) in sources.items():
    rows = samples[label]
    fields = [field, 'answer'] if label == 'gsm8k' else [field]
    for name in fields:
        texts = [r.get(name) if isinstance(r.get(name), str) else '' for r in rows]
        nonempty = [t for t in texts if t.strip()]
        counts = Counter(content_key(t) for t in nonempty)
        quality.append({'source': label, 'field': name, 'examined': len(rows),
                        'missing_or_empty': len(rows)-len(nonempty),
                        'duplicate_rows_after_first': sum(n-1 for n in counts.values()),
                        'replacement_char_rows': sum(chr(0xfffd) in t for t in texts),
                        'html_marker_rows': sum(bool(re.search(r'<(?:html|script|div|body)\b', t, re.I)) for t in texts)})
        values = [len(t) for t in texts]
        text_lengths[label + '.' + name] = values
        lengths.append({'source': label, 'field': name, 'unit': 'characters', **length_stats(values)})
table(quality)
table(lengths)
gsm_parsed = [extract_gsm_answer(r['answer']) for r in samples['gsm8k']]
print('GSM8K answer extraction:', sum(x is not None for x in gsm_parsed), '/', len(gsm_parsed))

# Show examples instead of inferring semantic quality from length alone.
for label, (_, _, field) in sources.items():
    display(HTML('<h4>' + html.escape(label) + ': short / median / long examples</h4>'))
    ordered = sorted(samples[label], key=lambda r: len(str(r.get(field) or '')))
    if ordered:
        table([{'position': tag, 'characters': len(str(row.get(field) or '')),
                'text_preview': str(row.get(field) or '')[:700]}
               for tag, row in [('short', ordered[0]), ('median', ordered[len(ordered)//2]), ('long', ordered[-1])]])


source,field,examined,missing_or_empty,duplicate_rows_after_first,replacement_char_rows,html_marker_rows
fineweb,text,1000,0,0,1,0
math,text,1000,0,0,1,0
gsm8k,question,7473,0,0,0,0
gsm8k,answer,7473,0,0,0,0


source,field,unit,count,min,mean,p50,p95,p99,max
fineweb,text,characters,1000,286,4428.41,2758,13226,29451,94830
math,text,characters,1000,101,7100.08,2612,30008,60727,285471
gsm8k,question,characters,7473,42,234.51,217,414,527,985
gsm8k,answer,characters,7473,50,287.5,257,562,749,1228


GSM8K answer extraction: 7473 / 7473


position,characters,text_preview
short,286,"Bluebulb Projects' The Measure of Things presents: 2,100 is about three times the number of Kernels on an Ear of Corn. In other words, the count of Kernels on an Ear of Corn is 0.30 times that amount. (Corn, a.k.a. Maize; Kernel a.k.a. Seed) (average) (for Dent Corn, a.k.a. field corn)"
median,2758,"We are all different in the way we prefer to learn. Some students like to work in groups, and others prefer to work alone. Some students like to discuss things over Skype or messenger and some like to work together in person. There is a reason why we have the saying ‘two heads are better than one’, so let’s see how this applies to different types of students. If you think you always prefer to work alone: - It is great that you are very self-sufficient. Students who prefer to work alone are often confident in their own abilities. However sometimes these students make it more difficult for themselves by not asking for help when they need it. So if you are this type of student, don’t struggle o"
long,94830,"From the Highlands of Lemuria Theosophical Quarterly, April, July, October, 1917 & January, April, 1918 Alfred Russel Wallace has dedicated one of the most charming chapters of Island Life, the most delightful and fascinating of all his books, to the plants and living creatures of the Hawaiian islands, the first stopping-place, going westward from California. Both plants and animals, he says, are among the most interesting in the world, perhaps the best example in the world of the flora and fauna of a genuine oceanic island, an island, that is, which appears never to have been united by land to any of the existing continents. For there are many islands which are not true islands, in this sen"


position,characters,text_preview
short,101,#### 4-C1.0.2 Describe the purposes of government as identified in the Preamble of the Constitution.
median,2624,"# Test that at least one hazard is greater than hazard in control I have an experiment with one control and three experimental arms, A, B and C. My experiment generates survival data. I have survival times for each group, $$T_{tj}$$, where $$t = \{ control, A, B, C\}$$ and $$j = 1, \dots, t_j.$$ I want to know if at least one arm has a lower hazard than the control arm. Denote the hazards with $$\lambda_t$$ I know hazards can be functions, so I can assume the hazards are proportional (preferred) or constant over time. The hypothesis test I'm interested in is: $$H_0 \lambda_j = \lambda_k$$ for all $$(j,k)$$ against, $$H_a: \lambda_{control} < \lambda{j}$$ for some $$j$$. The alternative can"
long,285471,"# 10 Best RV Freshwater Systems - Updated August 2022 15,559 Reviews Scanned And Analyzed Rank Image Product Name Score Check Price 1 Camco 22813 4ft Premium Drinking Water Hose - Lead and BPA Free, Anti-Kink Design, 20% Thicker Than Standard Hoses 5/8… Camco 9.8 Score View Product 2 Renator M11-0660R Water Pressure Regulator Valve. Brass Lead-free Adjustable Water Pressure Reducer with Gauge for RV… Renator 9.6 Score View Product 3 Camco Water Heater Tank Rinser-Cleanses and Removes Sediment that Collects at the Bottom of RV Water Heater, Extends… Camco 9.2 Score View Product 4 ONENESS 369 (2 Pack) Suburban Atwood RV, Camper, Travel Trailer Anode Rod Replacement Part - Magnesium with PTFE"


position,characters,text_preview
short,42,What is fifteen more than a quarter of 48?
median,217,"Calvin is a bug collector. In his collection, he has 12 giant roaches, 3 scorpions, half as many crickets as roaches, and twice as many caterpillars as scorpions. How many insects does Calvin have in his collection?"
long,985,"Hasan is packing up his apartment because he’s moving across the country for a new job. He needs to ship several boxes to his new home. The movers have asked that Hasan avoid putting more than a certain weight in pounds in any cardboard box. The moving company has helpfully provided Hasan with a digital scale that will alert him if a package is too heavy. Hasan is in the kitchen, and he fills a cardboard box with 38 dinner plates. When he checks the box, the scale reports his box is too heavy. Hasan knows each of his plates weighs 10 ounces. He removes a single plate from the box and checks the movers’ scale again. The scale reports his box is still too heavy. Hasan repeats the process again"


## 3. Biểu đồ phân bố độ dài

Biểu đồ dùng các khoảng tăng theo log để thấy cả chuỗi ngắn và chuỗi rất dài. Trục dọc là số bản ghi trong phạm vi khảo sát.


In [4]:
def histogram_svg(values, title):
    edges = [0, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, math.inf]
    counts = [sum(a <= x < b for x in values) for a, b in zip(edges, edges[1:])]
    peak = max(counts, default=1) or 1
    bars = []
    for i, count in enumerate(counts):
        x, height = 40 + i*70, 180*count/peak
        label = str(edges[i]) + '+' if math.isinf(edges[i+1]) else str(edges[i]) + '-' + str(edges[i+1]-1)
        bars.append(f'<rect x="{x}" y="{230-height}" width="55" height="{height}" fill="#377eb8"/><text x="{x+27}" y="{220-height}" text-anchor="middle">{count}</text><text x="{x+27}" y="250" text-anchor="middle" font-size="9">{label}</text>')
    svg = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 700 285"><rect width="700" height="285" fill="white"/><text x="20" y="24">' + html.escape(title) + '</text>' + ''.join(bars) + '<text x="250" y="275">Length (characters); bar labels = record counts</text></svg>'
    return svg

for label, values in text_lengths.items():
    svg = histogram_svg(values, label)
    (EDA_DIR / (label + '.svg')).write_text(svg, encoding='utf-8')
    display(HTML(svg))


## 4. GSM8K: chia tập và tác động lên SFT

Phân tích toàn bộ train, giữ 500 câu validation theo hash/seed như pipeline. Không đọc nội dung test ở bước này.
Nếu đã có tokenizer dự án, đo số token và tỷ lệ vượt 512/1024; nếu chưa có thì bỏ qua, không quy đổi ký tự thành token.
Đây là phép đo theo tokenizer hiện có: phải đo lại khi thay tokenizer chính thức.


In [5]:
from llm_reasoning.data_prep import split_gsm8k
from llm_reasoning.sft_data import encode_example
from llm_reasoning.tokenizer import ByteBPETokenizer

split = split_gsm8k(samples['gsm8k'], [], revision=revisions['gsm8k'], seed=SEED, validation_count=500)
table([{'split': 'gsm_train_before_filter', 'rows': len(split.train)},
       {'split': 'gsm_validation', 'rows': len(split.validation)}])
token_summary = []
tokenizer_path = ROOT / 'data' / 'pretrain' / 'tokenizer.json'
if tokenizer_path.exists():
    import hashlib
    tokenizer = ByteBPETokenizer.load(tokenizer_path)
    print('Tokenizer SHA256:', hashlib.sha256(tokenizer_path.read_bytes()).hexdigest())
    for name, rows in [('train', split.train), ('validation', split.validation)]:
        sizes, short, invalid = [], 0, 0
        for row in rows:
            parsed = extract_gsm_answer(row['answer'])
            if parsed is None:
                invalid += 1
                continue
            solution, answer = parsed
            if len(solution.split()) < 4:
                short += 1
                continue
            example = encode_example(tokenizer, source='gsm8k', question=row['question'],
                                     solution=solution, answer=answer, max_seq_len=10**12)
            sizes.append(example['token_count'])
        token_summary.append({'split': name, 'unit': 'project BPE tokens', **length_stats(sizes),
                              'invalid_answer': invalid, 'too_short_solution': short,
                              'over_512': sum(x > 512 for x in sizes), 'over_1024': sum(x > 1024 for x in sizes),
                              'accepted_tokens_at_1024': sum(x for x in sizes if x <= 1024)})
    table(token_summary)
else:
    print('No project tokenizer yet: run notebook 02, then rerun this section for actual BPE lengths.')


split,rows
gsm_train_before_filter,6973
gsm_validation,500


Tokenizer SHA256: 47eac6bc122ce0d17d6cccdcc1745059f9a65efe125867528500ddf58b1cec7e


split,unit,count,min,mean,p50,p95,p99,max,invalid_answer,too_short_solution,over_512,over_1024,accepted_tokens_at_1024
train,project BPE tokens,6969,70,260.64,243,442,546,831,0,4,121,0,1816382
validation,project BPE tokens,500,83,262.12,243,453,553,612,0,0,8,0,131059


## 5. Lưu bằng chứng và kết luận review

File JSON chứa revision, phạm vi khảo sát và các thống kê. SVG có thể dùng trong báo cáo.
Khi đọc kết quả, ghi nhận:

- Quy mô subset nguồn và số tài liệu khảo sát có khác nhau không?
- Dữ liệu có bản rỗng, trùng hoặc trang toán lạc đề không? Xem mẫu trước khi đổi bộ lọc.
- p95/p99 có cho thấy nhiều chuỗi dài? Bao nhiêu câu GSM8K bị loại tại giới hạn 1024 token?
- Ngân sách 80M/20M token là mục tiêu chuẩn bị; số thực tế phải lấy từ manifest notebook 02.
- EDA không chứng minh lời giải đúng về toán học; không dùng test để sửa dữ liệu hoặc chọn cấu hình.


In [6]:
summary = {'seed': SEED, 'sample_size_limit_pretrain': SAMPLE_SIZE,
           'shuffle_buffer': SHUFFLE_BUFFER, 'inventory': inventory,
           'examined_rows': {k: len(v) for k, v in samples.items()},
           'quality': quality, 'character_lengths': lengths,
           'gsm_answer_parse_success': sum(x is not None for x in gsm_parsed),
           'gsm_token_lengths': token_summary}
summary_path = EDA_DIR / 'summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Saved:', summary_path)
print('Charts:', EDA_DIR)


Saved: d:\WORKSPACE\Project\llm-reasoning\artifacts\eda\summary.json
Charts: d:\WORKSPACE\Project\llm-reasoning\artifacts\eda
